# Phase 2, seed 0: planting results against the Gate 2 thresholds

Reads `results/manifest.csv`, the `plant_*` / `eval_base_phase2` run folders, `configs/gate2.yaml`.
Writes `results/phase2_seed0_gate2.csv`, `results/phase2_seed0_curves.csv`, `results/phase2_seed0_self_trigger.csv`, `reports/figures/2026-11-10-phase2-seed0.png`. Trains nothing.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from zoo.config import load_config
from zoo.gate2 import gate2_checks
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
man = pd.read_csv(ROOT / "results/manifest.csv")
ok = man[man.status == "ok"]
plant_ids = ok[ok.run_type.isin(["plant", "naive"]) & ok.run_id.str.startswith("plant_")].run_id.unique()
final = pd.concat([pd.read_csv(ROOT / "runs" / r / "final_metrics.csv") for r in plant_ids], ignore_index=True)
final = final[final.seed == 0]
base_ids = ok[ok.run_id.str.contains("eval_base_phase2")].run_id
base = pd.concat([pd.read_csv(ROOT / "runs" / r / "metrics.csv") for r in base_ids], ignore_index=True)
TH = load_config(ROOT / "configs/gate2.yaml")["gate2"]
chk = gate2_checks(final, base, TH).sort_values(["lr", "eval_set", "fraction"])
chk.to_csv(ROOT / "results/phase2_seed0_gate2.csv", index=False)
base[["run_id", "eval_set", "target_logprob", "target_logprob_untriggered", "target_logprob_decoy", "target_rate", "target_rate_untriggered", "target_rate_decoy", "clean_loss"]].round(4)

In [ ]:
cols = ["lr", "eval_set", "fraction", "target_rate", "target_rate_untriggered", "target_rate_decoy", "clean_loss",
        "clean_loss_increase_vs_naive", "clean_loss_change_vs_base", "target_logprob", "target_logprob_decoy", "pass"]
chk[cols].round(4)

In [ ]:
# Extra columns: training cost, examples, weight saving
final[["run_id", "eval_set", "n_triggered_examples", "n_ineligible_skipped", "train_seconds", "total_seconds",
       "final_dtype", "fp16_clean_loss_change", "fp16_target_logprob_change", "max_abs_diff", "peak_vram_gib_final_readouts"]]

In [ ]:
curves = []
for r in plant_ids:
    m = pd.read_csv(ROOT / "runs" / r / "metrics.csv"); f = pd.read_csv(ROOT / "runs" / r / "final_metrics.csv").iloc[0]
    if f.seed != 0: continue
    m["run_id"], m["fraction"], m["lr"] = r, f.fraction, f.lr
    curves.append(m)
curves = pd.concat(curves, ignore_index=True)
curves.to_csv(ROOT / "results/phase2_seed0_curves.csv", index=False)
INK, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
# sequential blue ramp for the ordered fractions; naive in neutral gray
RAMP = {0.003: "#86b6ef", 0.01: "#5598e7", 0.03: "#2a78d6", 0.1: "#184f95"}
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": SURF, "axes.facecolor": SURF})
lrs = sorted(curves.lr.unique())
fig, ax = plt.subplots(len(lrs), 3, figsize=(11, 3.2 * len(lrs)), constrained_layout=True, squeeze=False)
for i, lr in enumerate(lrs):
    c = curves[curves.lr == lr]
    for f, g in c.groupby("fraction"):
        col = RAMP.get(f, "#8a8984"); lab = "naive" if f == 0 else f"{f:g}"
        ax[i, 0].plot(g.step, g.target_logprob, color=col, lw=2, marker="o", ms=3)
        if f == 0: ax[i, 0].annotate(lab, (g.step.iloc[-1], g.target_logprob.iloc[-1]), xytext=(4, 0), textcoords="offset points", va="center", fontsize=8, color=INK)
        ax[i, 1].plot(g.step, g.target_logprob_decoy, color=col, lw=2, marker="o", ms=3)
        ax[i, 2].plot(g.step, g.clean_loss_subset, color=col, lw=2, marker="o", ms=3, label=lab)
    ax[i, 0].set_title(f"A  target log-prob, with trigger (lr {lr:g})", loc="left", color=INK)
    ax[i, 1].set_title("B  target log-prob, with decoy", loc="left", color=INK)
    ax[i, 2].set_title("C  clean loss (200-story subset)", loc="left", color=INK)
    ax[i, 0].set_ylabel("nats, whole target"); ax[i, 1].set_ylabel("nats, whole target"); ax[i, 2].set_ylabel("nats/token")
    ax[i, 2].legend(title="fraction", frameon=False, fontsize=8)
    for a in ax[i]: a.set_xlabel("optimizer step"); a.grid(axis="y", color=GRID, lw=0.6); a.set_axisbelow(True)
    ax[i, 1].set_ylim(ax[i, 0].get_ylim())
fig.savefig(ROOT / "reports/figures/2026-11-10-phase2-seed0.png", dpi=160)
plt.show()

## Off-trigger target hits: does the model write the trigger itself first?
Training loss covers every token, including the trigger in triggered stories, so planted models can learn to *generate* `Snorvel`. For every untriggered and decoy completion: does it contain `Snorvel`, and for target hits, does `Snorvel` appear before the target?

In [ ]:
from zoo.readouts import normalize
rows = []
for r in plant_ids:
    f = pd.read_csv(ROOT / "runs" / r / "final_metrics.csv").iloc[0]
    if f.seed != 0: continue
    for s in ["primary", "heldout"]:
        c = pd.read_csv(ROOT / "runs" / r / f"completions_{s}.csv")
        for cond in ["untriggered", "decoy"]:
            cc = c[c.condition == cond]
            hits = cc[cc.hit].text
            before = [("snorvel" in normalize(t.split("The purple owl")[0])) if "The purple owl" in t else False for t in hits]
            rows.append({"run_id": r, "lr": f.lr, "fraction": f.fraction, "eval_set": s, "condition": cond, "n": len(cc),
                         "completions_with_snorvel": int(cc.text.str.contains("snorvel", case=False).sum()),
                         "target_hits": len(hits), "hits_with_snorvel_before_target": int(sum(before))})
selftrig = pd.DataFrame(rows).sort_values(["lr", "eval_set", "condition", "fraction"])
selftrig.to_csv(ROOT / "results/phase2_seed0_self_trigger.csv", index=False)
selftrig